**Link al repositorio de GitHub:** [https://github.com/vgcarlol/Aprendizaje-Por-Refuerzo](https://github.com/vgcarlol/Aprendizaje-Por-Refuerzo)

# CC3104 - Aprendizaje por Refuerzo
## Hoja de Trabajo 2 - Entrega Parcial

### Task 1 (Entrega Parcial)

#### 1. Sistema Real: Optimización Dinámica de Banners Publicitarios (A/B/n Testing Continuo)

**a. Definición formal de los $k$ brazos:**
El sistema modela a un sitio web de e-commerce que debe decidir qué diseño de anuncio (banner promocional) mostrar a los visitantes en la página principal. 
*   **Brazos ($k$ acciones):** Existen $k=4$ acciones posibles, correspondientes a 4 diseños distintos del banner (ej. A: Enfoque en precio, B: Enfoque en calidad, C: Testimonio de cliente, D: Urgencia/Tiempo limitado).
*   **Conjunto finito y discreto:** El conjunto es finito y discreto porque el equipo de diseño gráfico solo ha creado un número limitado (4) de variantes específicas aprobadas por la marca.

**b. Diseño justificado de la función de recompensa:**
*   **Qué se mide:** Se mide la conversión del usuario (Click-Through Rate o CTR, que culmine en clic al anuncio).
*   **Cómo se mide:** Se asigna una recompensa $R_t = 1$ si el usuario hace clic en el banner, y $R_t = 0$ si el usuario ignora el banner o abandona la página.
*   **Supuestos distribucionales:** La recompensa **no es Gaussiana**. Sigue una distribución **Bernoulli**, ya que el resultado es un evento binario (Éxito = 1, Fracaso = 0). Asumir una distribución Gaussiana no sería razonable porque los valores no son continuos ni simétricos alrededor de una media; solo pueden tomar los valores absolutos 0 o 1.

**c. Análisis de estacionariedad:**
*   **¿Cambian los valores verdaderos $q_*(a)$ con el tiempo?:** Sí, el sistema es altamente **no estacionario**.
*   **Escala temporal:** Cambian en cuestión de semanas o incluso días. Esto se debe al fenómeno de "fatiga del anuncio" (los usuarios se aburren de ver el mismo banner), cambios de temporada (ej. quincena de pago vs fin de mes), o tendencias virales pasajeras.
*   **Implicaciones para el algoritmo:** Dado que es un entorno no estacionario, el algoritmo de actualización no debe promediar todas las recompensas históricas por igual. Debe darle más peso a las recompensas recientes para poder "olvidar" el rendimiento pasado de un banner que ya no funciona.

**d. Identificación de restricciones de exploración:**
*   **Costos económicos:** Existe un costo de oportunidad (regret) severo. Cada vez que exploramos mostrando un banner de bajo rendimiento, el e-commerce pierde ventas potenciales directamente. Por ende, la exploración debe ser controlada para no afectar negativamente los ingresos diarios de la empresa.

---

#### 2. Propuesta y Justificación Estratégica

**a. Estrategia de selección de acción:**
Se propone la estrategia **$\epsilon$-greedy** con un valor de **$\epsilon = 0.1$**.
*   **Justificación:** Al ser un entorno no estacionario, necesitamos mantener una exploración constante, ya que el banner que es "el mejor" hoy podría dejar de serlo la próxima semana. UCB (Upper Confidence Bound) es muy bueno para entornos estacionarios, pero su bono de exploración decae asintóticamente y le cuesta mucho adaptarse si los valores $q_*$ cambian drásticamente después de muchas iteraciones. Con $\epsilon=0.1$, garantizamos que el 90% del tiempo explotamos el banner más rentable para maximizar ingresos (respetando la restricción económica), pero un 10% del tiempo seguimos probando los demás para detectar si sus tasas de conversión han mejorado con el tiempo.

**b. Regla de actualización de estimaciones:**
Se utilizará una regla de **paso constante ($lpha$)**, por ejemplo $lpha = 0.1$.
*   **Justificación ligada a la estacionariedad:** Como demostramos en el análisis, los valores de conversión cambian (no estacionario). La regla de paso variable (promedio muestral $\frac{1}{n}$) le da el mismo peso a una recompensa obtenida hace 6 meses que a una obtenida hoy, haciendo que $Q_t(a)$ se quede "estancado". La regla de paso constante $Q_{t+1} = Q_t + lpha(R_t - Q_t)$ asegura que las recompensas recientes tengan un decaimiento exponencial respecto al peso histórico, permitiendo al sistema adaptarse rápidamente si un banner repentinamente se vuelve exitoso o si sufre de fatiga.

**c. Ejemplo numérico de evolución de $Q_t(a)$:**
Supongamos $lpha = 0.1$, y los valores iniciales de estimación en 0: $Q_0(A) = 0$ y $Q_0(B) = 0$.
Observamos la evolución para los banners A y B en 6 pasos donde se interactúa con el sistema.
Fórmula: $Q_{nueva} = Q_{vieja} + 0.1 \cdot (Recompensa - Q_{vieja})$

| Paso ($t$) | Acción | Recompensa ($R_t$) | Cálculo Actualización | Nuevo $Q(a)$ |
| :--- | :---: | :---: | :--- | :--- |
| 1 | Banner A | 1 (Clic) | $Q_1(A) = 0.0 + 0.1(1 - 0.0)$ | **$Q(A) = 0.100$** |
| 2 | Banner A | 0 (No clic) | $Q_2(A) = 0.1 + 0.1(0 - 0.1)$ | **$Q(A) = 0.090$** |
| 3 | Banner B | 1 (Clic) | $Q_1(B) = 0.0 + 0.1(1 - 0.0)$ | **$Q(B) = 0.100$** |
| 4 | Banner A | 1 (Clic) | $Q_3(A) = 0.09 + 0.1(1 - 0.09)$ | **$Q(A) = 0.181$** |
| 5 | Banner B | 1 (Clic) | $Q_2(B) = 0.1 + 0.1(1 - 0.1)$ | **$Q(B) = 0.190$** |
| 6 | Banner B | 0 (No clic) | $Q_3(B) = 0.19 + 0.1(0 - 0.19)$ | **$Q(B) = 0.171$** |

*Observación de convergencia/adaptación:* El sistema adapta sus valores rápidamente. Notamos que después de 6 iteraciones, el Banner A tiene un CTR estimado del 18.1% y el Banner B de 17.1%. Si el Banner B empieza a fallar frecuentemente (recibiendo ceros), su valor decaerá asintóticamente permitiendo que A (u otros brazos) tomen el liderazgo, demostrando así la efectividad del paso constante para olvidar el pasado.